### Idea:
Create an environment with maximum bounds, initialise a random straight line path, and place some random obstacles.

In [1]:
import numpy as np
import roboticstoolbox as rtb
import spatialgeometry as sg
import spatialmath as sm
import swift
import time
import matplotlib as mtb
import matplotlib.pyplot as plt

from typing import Sequence, Tuple, List, Dict


In [2]:
# World bounds


# Simulation parameters
# dt = 0.05
# simulation_time = 20.0  # seconds
# simulation_steps = int(simulation_time / dt)
world_bounds = (-2.5, 2.5, -2.5, 2.5)  # min_x, max_x, min_y, max_y

# # Launch Swift viewer
# env = swift.Swift()
# env.launch(realtime=True, comms="rtc", browser="notebook")
# time.sleep(1.0)



In [3]:
# Define colour variables, for simulation environment

# --- SIMULATION CONSTANTS & COLORS ---
FLOOR_HEIGHT = 0.02
COLOR_BLUE = (0.2, 0.5, 0.8, 1.0)   # Robot color
COLOR_WALL = (0.3, 0.3, 0.3, 1.0)   # Wall color
COLOR_RAY = (1.0, 0.8, 0.0, 0.7)    # Sensor ray color (Yellow)
COLOR_GREEN = (0.2, 0.8, 0.3, 0.7)  # Waypoint/Target color
COLOR_GREY = (0.5, 0.5, 0.5, 0.8)   # Path line color
COLOR_WHITE = (1.0, 1.0, 1.0, 0.6)  # World ground



In [4]:
# environment setup

env = swift.Swift()
env.launch(realtime=True,comms="rtc",browser="notebook")
#env.set_camera_pose([0,0,0])

# Simulation parameters
dt = 0.01  # time step
# time = np.arange(0, 10, dt)  # simulation time



In [5]:
# world bounds

world_bounds = (-2.5, 2.5, -2.5, 2.5)  # min_x, max_x, min_y, max_y


def add_world_bounds(env: swift.Swift, bounds: Tuple[float, float, float, float]) -> sg.Cuboid:
    """Draw a translucent floor tile highlighting the workspace."""
    min_x, max_x, min_y, max_y = bounds
    width, depth = max_x - min_x, max_y - min_y
    floor = sg.Cuboid(scale=[width, depth, FLOOR_HEIGHT], color=COLOR_WHITE)
    floor.T = sm.SE3((min_x + max_x) / 2.0, (min_y + max_y) / 2.0, 0.01)
    env.add(floor)
    return floor

In [6]:
add_world_bounds(env, world_bounds)


cuboid,
[0.   0.   0.01]

In [7]:

COLOR_RED = (0.8, 0.2, 0.3, 1.0)



def select_point_on_edge(edge_num, bounds):
    point = np.random.uniform(bounds[0], bounds[1])
    if 0 <= edge_num <= 1:
        return bounds[edge_num], point
    return point, bounds[edge_num]

def create_random_line(e, bounds):
    # line: ax+b
    # must pass through env
    # will be defined by points inside env
    random_edge_1 = np.random.randint(0,4)
    random_edge_2 = np.random.randint(0,4)
    while random_edge_1 == random_edge_2:
        random_edge_2 = np.random.randint(0,4)
    # select 2 separate edges to connect

    x_1, y_1 = select_point_on_edge(random_edge_1, bounds)
    x_2, y_2 = select_point_on_edge(random_edge_2, bounds)

    dx, dy = x_1 - x_2, y_1 - y_2
    length = np.hypot(dx, dy)
    angle = np.arctan2(dy, dx)
    cx, cy = (x_1 + x_2) / 2.0, (y_1 + y_2) / 2.0

    line = sg.Cuboid(scale=[length, 0.03, 0.01], color=COLOR_RED)
    line.T = sm.SE3(cx, cy, 0.015) * sm.SE3.Rz(angle)
    env.add(line)

create_random_line(env, world_bounds)


In [ ]:

def select_random_target(e, bounds):
    x = np.random.uniform(bounds[0], bounds[1])
    y = np.random.uniform(bounds[2], bounds[3])
    z = 0.15 # so not touching ground

    return x, y, z

def add_random_objects(e, bounds, obj_num):
    obj_positions = []

    for obj in range(obj_num):
        x, y, z = select_random_target(e, bounds)
        
        # maybe for future: make objects so they cannot overlap
        # otherwise you have simply created a larger object

        # while (x,y,z) in obj_positions:
        #     x, y, z = select_random_target(e, bounds)
        
        cube_size = np.random.uniform(0.25, 1) # later do this
        
        block = sg.Cuboid(scale=[cube_size] * 3, color=COLOR_BLUE) 
        block.T = sm.SE3(x, y, z)
        
        env.add(block)
    
add_random_objects(env, world_bounds, 3)

In [ ]:
# # Add walls

# def add_wall(e, cx, cy, w, h) -> sg.Cuboid:
#     """Helper function to add a static wall to the environment."""
#     wall = sg.Cuboid(scale=[w, h, 0.5], color=COLOR_WALL)
#     wall.T = sm.SE3(cx, cy, 0.25)
#     e.add(wall)
#     return wall

In [ ]:
# # Add environment walls

# # Build the L-Shaped Maze
# walls = []
# walls.append(add_wall(env, -1.0, 0.0, 0.2, 14.0)) # Left External Wall
# walls.append(add_wall(env,  1.0, 0.0, 0.2, 10.0)) # Right Internal Wall
# walls.append(add_wall(env, 4.0, 5.0, 6.2, 0.2))   # Bottom Horizontal Wall (The Obstacle)
# walls.append(add_wall(env, 4.0, 7.0, 10.2, 0.2))  # Top External Wall
# walls.append(add_wall(env, 9.2, 6.0, 0.2, 2.2))   # End Cap


In [ ]:
# # Add random obstacles

# def select_target(e, bounds):
#     x = np.random.random(bounds[0], bounds[1])
#     y = np.random.random(bounds[2], bounds[3])
#     z = 0.1 # so not touching ground

#     e.add()


# body_geom = sg.Cuboid(scale=[0.4, 0.2, 0.1], color=[0.2, 0.5, 0.8, 1.0])
# head_geom = sg.Cuboid(scale=[0.05, 0.05, 0.01], color=[0.8, 0.8, 0.8, 1.0])
# env.add(body_geom)
# env.add(head_geom)


# def add_obstacle()